# Sentiment Analysis with Large Language Models via OpenRouter

Large Language Models (LLMs) have demonstrated remarkable capabilities in text understanding tasks, including sentiment analysis. Unlike traditional approaches that require task-specific training, LLMs can perform sentiment analysis through **prompting** alone — either zero-shot or with few-shot examples.

## Why OpenRouter?

[OpenRouter](https://openrouter.ai/) provides a unified API that gives access to multiple LLM providers (OpenAI, Anthropic, Google, Meta, Mistral, etc.) through a single endpoint. This makes it easy to compare different models and switch between them without changing your code.

## Approach

We demonstrate three prompting strategies:
1. **Zero-shot classification** — direct sentiment prediction with no examples
2. **Few-shot classification** — providing labeled examples in the prompt
3. **Chain-of-thought reasoning** — asking the model to explain its reasoning before classifying

### Setup

You will need an OpenRouter API key. Sign up at [openrouter.ai](https://openrouter.ai/) and set your key as an environment variable:

```bash
export OPENROUTER_API_KEY="your-key-here"
```

In [ ]:
# pip install requests pandas tensorflow_datasets

## 1. OpenRouter API Client

In [ ]:
import os
import json
import requests
import time

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "your-key-here")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1/chat/completions"

def query_llm(prompt, model="meta-llama/llama-3.1-8b-instruct:free", temperature=0.0, max_tokens=100):
    """
    Send a prompt to an LLM via the OpenRouter API.
    
    Args:
        prompt: The user message to send
        model: OpenRouter model identifier
        temperature: Sampling temperature (0 = deterministic)
        max_tokens: Maximum response length
    
    Returns:
        The model's response text
    """
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    
    response = requests.post(OPENROUTER_BASE_URL, headers=headers, json=payload)
    response.raise_for_status()
    
    return response.json()["choices"][0]["message"]["content"].strip()

# Quick test
print(query_llm("Say 'hello' if you can hear me."))

## 2. Load Sample Data

In [ ]:
import pandas as pd
import tensorflow_datasets as tfds

dataset, info = tfds.load('imdb_reviews', with_info=True, as_supervised=True)
test_data = dataset['test']
test_df = tfds.as_dataframe(test_data, info)
test_df['text'] = test_df['text'].str.decode('utf-8')

# Use a small sample due to API rate limits and costs
SAMPLE_SIZE = 50
sample_df = test_df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

# Truncate long reviews to keep prompts manageable
sample_df['text_truncated'] = sample_df['text'].str[:500]

print(f"Sample size: {len(sample_df)}")
print(f"Label distribution:\n{sample_df['label'].value_counts()}")

## 3. Zero-Shot Sentiment Classification

The simplest approach: ask the model to classify the sentiment directly.

In [ ]:
def zero_shot_sentiment(text, model="meta-llama/llama-3.1-8b-instruct:free"):
    """Classify sentiment using zero-shot prompting."""
    prompt = f"""Classify the sentiment of the following movie review as either 'positive' or 'negative'.
Respond with ONLY one word: positive or negative.

Review: {text}

Sentiment:"""
    
    response = query_llm(prompt, model=model, max_tokens=10)
    
    # Parse response
    response_lower = response.lower().strip()
    if 'positive' in response_lower:
        return 1
    elif 'negative' in response_lower:
        return 0
    else:
        return -1  # Could not parse

# Run on sample
print("Running zero-shot classification...")
zero_shot_preds = []
for i, row in sample_df.iterrows():
    pred = zero_shot_sentiment(row['text_truncated'])
    zero_shot_preds.append(pred)
    if (i + 1) % 10 == 0:
        print(f"  Processed {i + 1}/{len(sample_df)}")
    time.sleep(0.5)  # Respect rate limits

sample_df['zero_shot_pred'] = zero_shot_preds
print("Done.")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Filter out unparseable responses
valid_mask = sample_df['zero_shot_pred'] != -1
print(f"Parseable responses: {valid_mask.sum()}/{len(sample_df)}")

if valid_mask.sum() > 0:
    acc = accuracy_score(sample_df.loc[valid_mask, 'label'], sample_df.loc[valid_mask, 'zero_shot_pred'])
    print(f"\nZero-Shot Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(
        sample_df.loc[valid_mask, 'label'],
        sample_df.loc[valid_mask, 'zero_shot_pred'],
        target_names=['Negative', 'Positive']
    ))

## 4. Few-Shot Sentiment Classification

Providing a few labeled examples in the prompt can improve accuracy and consistency.

In [ ]:
FEW_SHOT_EXAMPLES = """Example 1:
Review: "A masterfully crafted film with stunning performances and a deeply moving story."
Sentiment: positive

Example 2:
Review: "Painfully boring with wooden acting and a plot that goes nowhere."
Sentiment: negative

Example 3:
Review: "Despite a slow start, the film builds to a powerful and unforgettable climax."
Sentiment: positive

Example 4:
Review: "A complete waste of time. The script is lazy and the direction is uninspired."
Sentiment: negative"""


def few_shot_sentiment(text, model="meta-llama/llama-3.1-8b-instruct:free"):
    """Classify sentiment using few-shot prompting."""
    prompt = f"""Classify the sentiment of movie reviews as 'positive' or 'negative'.

{FEW_SHOT_EXAMPLES}

Now classify this review. Respond with ONLY one word: positive or negative.

Review: "{text}"
Sentiment:"""
    
    response = query_llm(prompt, model=model, max_tokens=10)
    response_lower = response.lower().strip()
    if 'positive' in response_lower:
        return 1
    elif 'negative' in response_lower:
        return 0
    else:
        return -1

# Run on sample
print("Running few-shot classification...")
few_shot_preds = []
for i, row in sample_df.iterrows():
    pred = few_shot_sentiment(row['text_truncated'])
    few_shot_preds.append(pred)
    if (i + 1) % 10 == 0:
        print(f"  Processed {i + 1}/{len(sample_df)}")
    time.sleep(0.5)

sample_df['few_shot_pred'] = few_shot_preds
print("Done.")

In [ ]:
valid_mask = sample_df['few_shot_pred'] != -1
print(f"Parseable responses: {valid_mask.sum()}/{len(sample_df)}")

if valid_mask.sum() > 0:
    acc = accuracy_score(sample_df.loc[valid_mask, 'label'], sample_df.loc[valid_mask, 'few_shot_pred'])
    print(f"\nFew-Shot Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(
        sample_df.loc[valid_mask, 'label'],
        sample_df.loc[valid_mask, 'few_shot_pred'],
        target_names=['Negative', 'Positive']
    ))

## 5. Chain-of-Thought Reasoning

Asking the model to reason step-by-step before giving its final answer can improve accuracy on ambiguous cases.

In [ ]:
def cot_sentiment(text, model="meta-llama/llama-3.1-8b-instruct:free"):
    """Classify sentiment using chain-of-thought prompting."""
    prompt = f"""Analyze the sentiment of the following movie review.

First, identify the key positive and negative aspects mentioned.
Then, determine the overall sentiment.
Finally, state your classification as EXACTLY one of: POSITIVE or NEGATIVE.

Review: "{text}"

Analysis:"""
    
    response = query_llm(prompt, model=model, max_tokens=300)
    
    # Extract the final classification from the response
    response_lower = response.lower()
    
    # Look for the last occurrence of positive/negative
    last_pos = response_lower.rfind('positive')
    last_neg = response_lower.rfind('negative')
    
    if last_pos > last_neg:
        return 1, response
    elif last_neg > last_pos:
        return 0, response
    else:
        return -1, response

# Run on a smaller subset for CoT (more tokens per request)
COT_SIZE = 20
cot_sample = sample_df.head(COT_SIZE).copy()

print("Running chain-of-thought classification...")
cot_preds = []
cot_explanations = []
for i, row in cot_sample.iterrows():
    pred, explanation = cot_sentiment(row['text_truncated'])
    cot_preds.append(pred)
    cot_explanations.append(explanation)
    if (i + 1) % 5 == 0:
        print(f"  Processed {i + 1}/{COT_SIZE}")
    time.sleep(0.5)

cot_sample['cot_pred'] = cot_preds
cot_sample['cot_explanation'] = cot_explanations
print("Done.")

In [ ]:
# Show a few examples with explanations
for i in range(min(3, len(cot_sample))):
    row = cot_sample.iloc[i]
    print(f"--- Review {i+1} (True label: {'Positive' if row['label'] == 1 else 'Negative'}) ---")
    print(f"Text: {row['text_truncated'][:200]}...")
    print(f"\nLLM Analysis:\n{row['cot_explanation']}")
    print()

In [ ]:
valid_mask = cot_sample['cot_pred'] != -1
if valid_mask.sum() > 0:
    acc = accuracy_score(cot_sample.loc[valid_mask, 'label'], cot_sample.loc[valid_mask, 'cot_pred'])
    print(f"Chain-of-Thought Accuracy: {acc:.4f} (on {valid_mask.sum()} samples)")

## 6. Comparing Models via OpenRouter

One major advantage of OpenRouter is the ability to compare different LLMs easily. Below we demonstrate how to evaluate the same prompt across multiple models.

In [ ]:
# Models available on OpenRouter (check openrouter.ai/models for current list)
# Some models offer free tiers; others require credits
MODELS_TO_COMPARE = [
    "meta-llama/llama-3.1-8b-instruct:free",
    "google/gemma-2-9b-it:free",
    # Add more models as needed — some popular options:
    # "openai/gpt-4o-mini",
    # "anthropic/claude-3.5-haiku",
    # "mistralai/mistral-small",
]

COMPARISON_SIZE = 20
comparison_sample = sample_df.head(COMPARISON_SIZE).copy()

results = {}
for model_name in MODELS_TO_COMPARE:
    print(f"\nEvaluating: {model_name}")
    preds = []
    for i, row in comparison_sample.iterrows():
        try:
            pred = zero_shot_sentiment(row['text_truncated'], model=model_name)
            preds.append(pred)
        except Exception as e:
            print(f"  Error on sample {i}: {e}")
            preds.append(-1)
        time.sleep(0.5)
    
    valid = [p != -1 for p in preds]
    valid_preds = [p for p, v in zip(preds, valid) if v]
    valid_labels = [l for l, v in zip(comparison_sample['label'], valid) if v]
    
    if valid_preds:
        acc = accuracy_score(valid_labels, valid_preds)
        results[model_name] = acc
        print(f"  Accuracy: {acc:.4f} ({len(valid_preds)} valid responses)")
    else:
        print("  No valid responses.")

print("\n=== Summary ===")
for model, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {model}: {acc:.4f}")

## Summary & Discussion

### Advantages of LLM-Based Sentiment Analysis
- **No training required**: Works out of the box with zero-shot or few-shot prompting
- **Handles nuance**: LLMs understand sarcasm, context, and mixed sentiments better than lexicon-based methods
- **Explainability**: Chain-of-thought prompting provides human-readable reasoning
- **Flexibility**: Easy to adapt to new domains by modifying prompts

### Limitations
- **Cost**: API calls incur per-token charges (though free tiers exist)
- **Latency**: Much slower than local models (VADER, SBERT)
- **Reproducibility**: Non-deterministic outputs even at temperature=0 (implementation-dependent)
- **Rate limits**: Not practical for large-scale batch processing without careful management
- **Privacy**: Text data is sent to external servers

### When to Use LLMs for Sentiment Analysis
- Small datasets where training a model is impractical
- When you need explainable predictions
- Complex or domain-specific texts where simpler methods fall short
- Rapid prototyping before investing in custom model training

### References
- Brown, T., et al. (2020). Language Models are Few-Shot Learners. *NeurIPS*.
- Wei, J., et al. (2022). Chain-of-Thought Prompting Elicits Reasoning in Large Language Models. *NeurIPS*.